# Agentic AI for Proactive QoS Assurance in 6G Network Slicing
### An Autonomous, Twin-Driven, Closed-Loop Control System — Project Overview

**Course context:** Computer Networks project, rebuilt from scratch. An earlier prototype used a PPO reinforcement-learning agent to manage 6G network slicing; this version replaces it with a **Digital Twin -> Probabilistic Forecaster -> LLM Planner -> Safety Layer -> Actuation** pipeline, keeping the old RL agent only as a comparison baseline.

This notebook is the entry point of the repository. It lays out the **problem statement**, **why 6G specifically**, **why an agentic LLM pipeline instead of plain RL or a fixed policy**, the **models used and why**, the **full system architecture**, the **project objectives**, and **what success looks like**.

> The agent itself — its prompt contract, its structured output, and the two-stage safety check that guards it — is covered separately and in full depth in `01_agent_overview.ipynb`, since it is the core novelty of this project rather than a general background concept.


## 1. Problem Statement

- **The mechanism:** 6G networks use **network slicing** to carve one physical network into several virtual ones, each tuned to a different traffic type:
  - **URLLC** (Ultra-Reliable Low-Latency Communication) — self-driving vehicles, remote surgery, industrial robotics. A few milliseconds of extra delay can cause a real, physical failure.
  - **eMBB** (Enhanced Mobile Broadband) — streaming, downloads. Tolerant of delay, no safety consequence.

- **The bottleneck:** Traditional slicing assigns each slice a **fixed** bandwidth split. That is fine at low load, but the instant demand spikes unpredictably, a rigid split cannot adapt — and the latency-critical URLLC slice breaches its QoS guarantee exactly when it matters most.

- **The reactive-control gap:** Even adaptive schemes that exist today are typically *reactive* — they respond only after congestion has already begun. This project targets the gap one step earlier: **predicting** the spike and **reallocating before it happens**, verified by an explicit safety check before anything touches the network.


## 2. Why 6G Specifically

- 6G's target URLLC latencies push toward the **sub-millisecond** range — tighter than 5G by roughly an order of magnitude, so the margin for a wrong or late allocation decision is much smaller.
- **"AI-native" self-management** is a stated core design goal of 6G standardisation, not a bolt-on feature added after the fact — this project is aligned with that direction rather than retrofitting AI onto a 5G-style network.
- **Network Digital Twins** are an actively researched 6G enabler specifically because they let an AI agent be trained and stress-tested safely, at zero risk, before anything resembling deployment — which is exactly the role the Digital Twin plays here (Stage 1 below).


## 3. Why Not Just Reinforcement Learning?

The earlier prototype for this same project used a **PPO** reinforcement-learning agent to decide slice allocations directly. It worked, but had two structural problems that motivated this rebuild — and the old agent is **kept in this repo**, not deleted, so it can be run as a second baseline in the final comparison:

- **Opaque decisions.** An RL policy's action can only be *observed*, never *inspected*. There is no way to ask it "why did you allocate it this way?" — which is a serious problem for a system a NOC engineer is expected to trust and eventually sign off on.
- **Purely reactive.** An RL agent conditioned only on the current state reacts *after* congestion is already visible in that state. It has no explicit forecast of what is about to happen.

This project's pipeline replaces the RL policy with an **LLM planner** that reasons in natural language over an explicit **forecast with uncertainty**, and every one of its decisions is checked by a **deterministic safety layer** before it is ever allowed to act — trading a black box for something proactive, explainable, and independently verified. The final evaluation (Stage 6) is a genuine **three-way comparison**: Static Baseline vs. legacy PPO/RL agent vs. the new Live Agentic System — not just agent-vs-baseline.


## 4. Models & Algorithms Used, and Why

| # | Model / Component | Why it's used here (one line) |
|---|---|---|
| 1 | **Non-linear Digital Twin** (mechanistic simulator, not learned) | Generates a network hard enough to be meaningful to control: a congestion-cliff latency curve, *correlated* wireless fading, and cross-slice resource contention — so a naive agent can't cheat by learning a trivial pattern. |
| 2 | **Probabilistic LSTM Forecaster** | Predicts each slice's near-term traffic as a **mean + variance**, not a single point estimate, so the planner downstream can make *risk-aware* decisions instead of reacting blindly to a possibly-wrong forecast. |
| 3 | **LLM Agentic Planner** | Reasons over the current state, the forecast, and the network's hard rules the way a human NOC operator would, and outputs a structured, inspectable allocation plan in plain language instead of an opaque set of weights. |
| 4 | **Rule-Based Safety / Validator Layer** (deterministic, *not* learned) | Every plan is checked for schema validity and physical/operational constraint satisfaction *before* it can touch the network. This is deliberately non-learned — safety cannot be allowed to depend on a model's behaviour. |
| 5 | **Legacy PPO Reinforcement-Learning Agent** (retained from the earlier prototype) | Kept only as a **second comparison baseline**, to empirically demonstrate the new pipeline's advantage rather than merely asserting it. |

Full design detail on model #3 and #4 together (they operate as one control step) is in `01_agent_overview.ipynb`.


### 4.1 The Congestion Cliff — What Makes the Twin Non-Trivial

A naive twin would model latency as growing linearly with load, which any simple controller can predict perfectly. Instead, this project's twin uses a **non-linear "congestion cliff"**: latency stays low and stable until load approaches capacity, then degrades sharply — much closer to how real queuing systems behave near saturation.

$$
\text{QueuingDelay}(L, C) \;\propto\; \left(\frac{L}{C}\right)^{\!11/5}
$$

where $L$ is offered load and $C$ is slice capacity. The exponent means a small increase in load near capacity produces a disproportionately large latency jump — this is the "cliff" the forecaster has to see coming, and the planner has to reallocate *before* the twin walks off it.


In [ ]:
# ---- Demonstration: why a fixed allocation fails near the congestion cliff ----
# This is a standalone illustration of the formula above, not the real twin implementation
# (the real, stateful twin — with fading + contention — lives in src/digital_twin.py).

import numpy as np
import matplotlib.pyplot as plt

def queuing_delay(load, capacity, exponent=11/5, base_delay=1.0):
    ratio = np.clip(load / capacity, 0, 0.999)  # avoid divide-by-zero right at the cliff edge
    return base_delay / (1 - ratio) ** 0  # placeholder avoided; see cliff formula below

def congestion_cliff_delay(load, capacity, exponent=11/5, floor_ms=1.0, scale=50.0):
    ratio = np.clip(load / capacity, 0, 1.5)
    return floor_ms + scale * ratio ** exponent

capacity = 100.0  # Mbps, arbitrary unit for illustration
loads = np.linspace(0, 130, 300)
delays = congestion_cliff_delay(loads, capacity)

fixed_alloc_load_example = 92.0  # a load a fixed 100 Mbps slice might see during a real spike
delay_at_example = congestion_cliff_delay(fixed_alloc_load_example, capacity)

plt.figure(figsize=(7, 4))
plt.plot(loads, delays, color='#1f4e8c')
plt.axvline(capacity, color='crimson', linestyle='--', linewidth=1, label='fixed capacity (100 Mbps)')
plt.scatter([fixed_alloc_load_example], [delay_at_example], color='crimson', zorder=5,
            label=f'load={fixed_alloc_load_example} -> delay={delay_at_example:.1f} ms')
plt.title('Congestion-cliff latency vs. load, at a fixed slice capacity')
plt.xlabel('Offered load (Mbps)'); plt.ylabel('Queuing delay (ms)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('congestion_cliff_demo.png', dpi=150)
plt.show()

print(f"At {fixed_alloc_load_example:.0f}/{capacity:.0f} Mbps load, delay is already "
      f"{delay_at_example:.1f} ms and rising steeply — this is the regime the forecaster "
      f"and planner must act *before*, not after.")


## 5. The Probabilistic Forecaster — Why Mean + Variance, Not Just a Point Forecast

A plain LSTM regressor predicts a single next value. This project's forecaster instead predicts a **Gaussian output** — a mean $\mu_t$ and a variance $\sigma_t^2$ — over the next-timestep traffic for each slice, trained by minimising the Gaussian negative log-likelihood rather than plain MSE:

$$
\mathcal{L}_{\text{NLL}} = \frac{1}{2}\log(2\pi\sigma_t^2) + \frac{(y_t - \mu_t)^2}{2\sigma_t^2}
$$

The reason this matters downstream: the **planner** can then act on a *risk-adjusted* forecast — e.g. plan against $\mu_t + k\sigma_t$ instead of $\mu_t$ alone when the slice is URLLC and $\sigma_t$ is high — rather than trusting a point estimate the twin's non-linearity can make dangerously wrong.


In [ ]:
# ---- Demonstration: why planning against mu + k*sigma matters more as forecast uncertainty grows ----
# Toy illustration only — the trained model lives in src/forecaster.py (Stage 2, later notebook).

import numpy as np

def risk_adjusted_target(mu, sigma, k=1.5):
    """Plan capacity against a conservative upper quantile, not the raw mean."""
    return mu + k * sigma

scenarios = [
    ("low-uncertainty forecast",  70.0, 3.0),
    ("moderate-uncertainty forecast", 70.0, 10.0),
    ("high-uncertainty forecast (e.g. just before a spike)", 70.0, 22.0),
]

print(f"{'scenario':<55} {'mu':>6} {'sigma':>7} {'risk-adjusted target':>22}")
for name, mu, sigma in scenarios:
    target = risk_adjusted_target(mu, sigma)
    print(f"{name:<55} {mu:6.1f} {sigma:7.1f} {target:22.1f}")

print("\nSame point forecast (mu=70) each time -- a point-estimate-only planner would "
      "allocate identically in all three cases. The risk-adjusted target correctly asks for "
      "more headroom exactly when the forecaster itself is least confident.")


## 6. The Live Agentic Planner + Safety Layer — Summary Only

This is the core novelty of the project, so it gets its **own notebook**: `01_agent_overview.ipynb`. In summary:

- The planner receives the current state, the forecaster's mean+variance output, and the network's hard rules, and returns a **structured JSON allocation plan** with a plain-English `reasoning` field (this is what powers the roadmap's "agent reasoning" dashboard panel).
- Every plan passes a **two-stage safety check** — schema validity, then physical/operational constraints — before actuation.
- A failed check triggers a **bounded retry with the specific error fed back to the LLM**, and only after retries are exhausted does the system fall back to a deterministic, known-safe policy for that timestep. This retry loop is an explicit add-on over the naive "check once, fall back" design.

Implementation lives in `src/agent_planner.py` (the planner) and `src/safety_layer.py` (the validator) — full walkthrough in `01_agent_overview.ipynb`.


## 7. System Architecture

![System Architecture](../images/system_architecture.svg)

**Reading the diagram top to bottom:**

1. **Enhanced Digital Twin** — produces realistic per-slice traffic and latency every timestep (congestion cliff + correlated fading + shared contention).
2. **Probabilistic Forecaster (LSTM)** — 30-timestep window in, mean+variance forecast out.
3. **Live Agentic Planner (LLM)** — branches into the two comparison arms: **3A Static Baseline** (fixed split; the legacy PPO/RL agent is also run here as a second control arm) and **3B Live Agentic System** (this project's contribution).
4. **Safety Layer** — syntax check, then semantic/constraint check; failure triggers a bounded retry with error feedback before falling back to a deterministic safe policy.
5. **Actuation + Metrics Logging** — the verified plan is applied to the twin, per-slice latency/throughput/QoS-violation flags are logged, and state feeds back into Stage 1 for the next control cycle.
6. **Comparison & Validation** — Static vs. legacy RL vs. Live Agentic System, on P99 tail latency, QoS violation rate, and eMBB throughput trade-off; results either get accepted or sent back to tune the prompt/forecaster and re-run Stages 3–6.
7. **UI Dashboard** (sits on top of the whole pipeline) — live traffic/latency/allocation charts, the agent-reasoning panel, and a manual fault/spike-injection control for live demos.


## 8. Project Objectives

- **Objective 1:** Build an Enhanced Digital Twin with a non-linear congestion-cliff latency model, correlated wireless fading, and shared base-station contention — realistic enough that a naive controller cannot trivially predict it.
- **Objective 2:** Train a probabilistic LSTM forecaster that outputs a mean + variance traffic prediction per slice, evaluated on calibration (not just point accuracy).
- **Objective 3:** Build a live LLM planner that consumes the twin state + forecast + hard rules and outputs a structured, inspectable allocation plan with a plain-English rationale.
- **Objective 4:** Implement a two-stage (syntactic + semantic) safety layer with a bounded, feedback-driven retry loop and a deterministic fallback policy, so no unverified plan can ever reach the network.
- **Objective 5:** Run a genuine three-way comparison — Static Baseline vs. legacy PPO/RL agent vs. Live Agentic System — and report the trade-offs honestly, including any cost to eMBB throughput.
- **Objective 6 (stretch):** Ship the live dashboard with the real-time agent-reasoning panel and manual fault-injection control described in the roadmap.


## 9. Who This Is For

This is not an end-user product — the "user" is the network itself, and by extension:

- **Telecom operators**, who would deploy something like this inside their network core to manage slicing decisions live.
- **NOC (Network Operations Center) engineers**, whose current manual/reactive monitoring role this is designed to assist or eventually replace — which is exactly why the agent's decisions need to be *explainable*, not just accurate.
- **Indirectly, the businesses depending on the URLLC guarantee** — hospitals, factories, autonomous fleets — who never touch the system but are the entire reason it needs to work.


## 10. What Success Looks Like

Measured against the static, fixed-allocation baseline **and** the legacy PPO/RL agent, under identical, unpredictable synthetic traffic:

- **Reduction in P99 tail latency** for the URLLC slice — the worst-case user experience, not the average.
- **Reduction in QoS violation rate** — how often URLLC latency actually crosses the critical threshold.
- **An honest, quantified eMBB throughput trade-off** — the acceptable cost of protecting URLLC, reported plainly rather than hidden.
- **A working, bounded safety layer** — verified empirically by injecting malformed/unsafe LLM outputs and confirming the fallback policy always triggers correctly, never a system fault.


## 11. Repository Structure

```
6g-agentic-network-slicing/
├── README.md
├── requirements.txt
├── notebooks/
│   ├── 00_overview.ipynb          <- you are here
│   ├── 01_agent_overview.ipynb    <- planner + safety layer deep dive (no code, explanation only)
│   ├── 02_digital_twin.ipynb      <- Stage 1 build + validation
│   ├── 03_forecaster.ipynb        <- Stage 2 build + training + calibration
│   ├── 04_agentic_planner.ipynb   <- Stage 3 build + live LLM integration
│   ├── 05_safety_layer.ipynb      <- Stage 4 build + adversarial/malformed-output tests
│   ├── 06_baseline_comparison.ipynb  <- Stage 5/6: static + legacy RL harness, full 3-way comparison
│   └── 07_results_and_report.ipynb   <- Figures, tables, final write-up
├── src/
│   ├── digital_twin.py
│   ├── forecaster.py
│   ├── agent_planner.py
│   ├── safety_layer.py
│   ├── schemas.py
│   └── baseline_agents.py
├── images/
│   └── system_architecture.svg
├── results/
└── data/
```

Every notebook after this one follows the same pattern: **explanation and math in the notebook, real implementation in the matching `src/*.py` file** — so the notebooks stay readable as documentation, and the code stays reusable and testable on its own.


## 12. Full Project Roadmap

| Stage | Notebook | Source file(s) | What happens |
|---|---|---|---|
| 0 | `00_overview.ipynb` | — | Problem statement, why 6G, why not RL alone, models used, architecture, objectives |
| 1 | `01_agent_overview.ipynb` | `agent_planner.py`, `safety_layer.py`, `schemas.py` | Deep dive on the agent's design: prompt contract, output schema, two-stage safety check, retry loop |
| 2 | `02_digital_twin.ipynb` | `digital_twin.py` | Congestion-cliff latency, correlated fading, shared contention — the simulated network |
| 3 | `03_forecaster.ipynb` | `forecaster.py` | Probabilistic LSTM: training, calibration, mean+variance evaluation |
| 4 | `04_agentic_planner.ipynb` | `agent_planner.py` | Live LLM integration wired to the twin + forecaster, running plans end-to-end |
| 5 | `05_safety_layer.ipynb` | `safety_layer.py` | Constraint checks, adversarial malformed-plan tests, fallback-policy verification |
| 6 | `06_baseline_comparison.ipynb` | `baseline_agents.py` | Static baseline + legacy PPO/RL agent harness, full three-way comparison run |
| 7 | `07_results_and_report.ipynb` | — | P99 latency, QoS violation rate, throughput trade-off figures; final report |

---
*Next: → `01_agent_overview.ipynb`*
